# SNN Framework Benchmark — Colab Runner (DVS128 Gesture)

Runs `docs/results/run_benchmark.py` (all 4 frameworks — Norse, snnTorch, SpikingJelly, Sinabs) on DVS128 Gesture, full-scale: 10 epochs, the entire train set, the entire test set, plus a full adversarial-robustness pass (FGSM + PGD-20) per framework — same as `learning/main.py`'s single-framework run, just looped across all four.

No Google Drive mounting here — everything stays on this Colab session's local disk (`/content/...`). If the session ends before you've downloaded results, they're gone, so download `docs/results/data/` and `docs/results/plots/` (last cell) before closing the tab.

**Before you run this — read this cell.** A full run is 4 frameworks x 10 epochs x the entire DVS128 Gesture train set + full test set + a full adversarial-robustness sweep (FGSM + PGD-20 across 4 epsilon values each = ~85 full test-set passes) per framework. This is many hours, likely most of a day. Keep the tab open and interact with it occasionally — Colab's free tier disconnects idle sessions, not just long-running ones.

Runtime -> Change runtime type -> GPU, before running anything below.

## 1. Get the codebase

Clones from the repo referenced in this project's own docs (`docs/roadmap.md`). If your remote/branch differs, edit the URL/branch below before running.

In [ ]:
REPO_URL = "https://github.com/Zuzu3290/SNNs-auf-GPUs.git"
BRANCH = "46-cache_engine"  # the branch this benchmark work is actually on -- main is 7 commits behind and doesn't have it

!git clone --branch {BRANCH} {REPO_URL} /content/SNNs-auf-GPUs
%cd /content/SNNs-auf-GPUs

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

import torch
print("torch:", torch.__version__, " CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected — check Runtime > Change runtime type > GPU")

## 3. Run the full benchmark

`--full` = 10 epochs, entire dataset, entire test set, plus the adversarial-robustness pass per framework (same steps `main.py` runs for one framework, looped over all four here). TRADES is off by default — pass `--trades` if you want adversarial training included instead.

In [ ]:
!python docs/results/run_benchmark.py --dataset "DVS128 Gesture" --full

## 4. Generate plots

In [ ]:
!python docs/results/make_plots.py --dataset "DVS128 Gesture"

## 5. View the tabular results (one row per framework)

In [ ]:
import pandas as pd

df = pd.read_csv("docs/results/data/dvs128_gesture/runs.csv")
pd.set_option("display.max_columns", None)
df

## 6. View the adversarial-robustness results per framework

In [ ]:
import glob

for path in sorted(glob.glob("docs/results/data/dvs128_gesture/*_adversarial.csv")):
    print(f"\n=== {path} ===")
    display(pd.read_csv(path))

## 7. View the plots inline

In [ ]:
from IPython.display import Image, display

for path in sorted(glob.glob("docs/results/plots/dvs128_gesture/*.png")):
    print(path)
    display(Image(filename=path))

## 8. Download everything before the session ends

No Drive mounting — zips the results and triggers a browser download instead.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/snn_benchmark_results", "zip", "docs/results", "data")
files.download("/content/snn_benchmark_results.zip")